# 실습 6 (심화) · LLM으로 탐지 근거 설명

탐지된 연결 한 줄을 LLM에게 설명하게 하고, 그 설명이 로그와 맞는지 검증합니다.
- 공격 유형은 DoS · Exploits · Reconnaissance · Normal 중 하나만 고르게 합니다.
- `evidence`에 적은 '필드=값'이 실제 로그에 있는지 코드로 확인합니다(지어낸 근거 찾기).
- 정답은 `../data/sample_logs_truth.csv`에서 따로 읽어 비교에만 씁니다.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "llm_client.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find project root containing llm_client.py")
sys.path.insert(0, str(PROJECT_ROOT))

from llm_client import create_llm_client

PROVIDER = "mono"
client, MODEL = create_llm_client(provider=PROVIDER, api_key_env="MY_API_KEY")
print(PROVIDER, MODEL)

In [ ]:
from openai.types.chat import ChatCompletionMessageParam

def ask(
    prompt: str,
    system: str | None = None,
    temperature: float = 0.2,
    show: bool = True,
) -> str:
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages: list[ChatCompletionMessageParam] = [
        {"role": "user", "content": prompt}
    ]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    if PROVIDER == "ollama":
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            stream=True,
            max_tokens=500,
        )
    else:
        stream = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            stream=True,
        )
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [3]:
import json, re

def parse_json(raw):
    """답에서 { ... } 부분만 꺼내 JSON으로 읽음. 없거나 깨졌으면 None"""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

In [4]:
TYPES = ["DoS", "Exploits", "Reconnaissance", "Normal"]
PROMPT = """당신은 금융기관 SOC 분석가입니다. 다음 네트워크 연결을 분석해 JSON으로만 답하세요.
evidence에는 판단에 쓴 필드와 값을 로그에 있는 그대로 '필드=값'으로 적고, 로그에 없는 정보는 추측하지 마세요.

필드 뜻: src/dst=출발지/목적지 IP:포트, service=서비스(-는 식별 안 됨), state=연결 상태(FIN 정상 종료, INT 응답 없음),
dur=지속시간(초), spkts/dpkts=송신/수신 패킷, sbytes/dbytes=송신/수신 바이트, rate=초당 패킷 수,
ct_dst_ltm=최근 100개 연결 중 같은 목적지 연결 수, ct_src_dport_ltm=최근 100개 연결 중 같은 출발지·목적지 포트 연결 수

연결: {flow}

{{"attack_type": "DoS / Exploits / Reconnaissance / Normal 중 하나",
  "mitre_tactic": "TA0043 Reconnaissance / TA0001 Initial Access / TA0040 Impact / 없음 중 하나",
  "kill_chain_stage": "초반(정찰) / 중반(침투) / 후반(피해) / 정상 중 하나",
  "evidence": ["필드=값", "..."],
  "reason": "한 줄 설명",
  "recommended_action": "권고 조치"}}"""

def explain(flow):
    for _ in range(2):
        d = parse_json(ask(PROMPT.format(flow=flow), system="JSON만 출력하는 보안 분석가."))
        if d and d.get("attack_type") in TYPES and isinstance(d.get("evidence"), list):
            return d
    return None                                       # 판정불가 → 사람 검토

## 정찰 · 침투 · DoS 각 1줄 (sample_logs.txt 8 · 12 · 1번째 줄)

In [5]:
import pandas as pd

LINES = open("sample_logs.txt", encoding="utf-8").read().splitlines()
truth = pd.read_csv("../data/sample_logs_truth.csv").set_index("line")["attack_cat"]

for n in [8, 12, 1]:
    flow = LINES[n - 1][28:]
    print(f"\n===== {n}번째 줄 (정답 {truth[n]}) =====\n{flow}\n")
    d = explain(flow)
    if d is None:
        print("\n→ 판정불가: 사람 검토")
        continue
    fake = [e for e in d["evidence"] if str(e).replace(" ", "") not in flow.replace(" ", "")]
    print(f"\n→ 예측 {d['attack_type']} / 정답 {truth[n]} | {d['mitre_tactic']} | {d['kill_chain_stage']}")
    print(f"→ 로그에 없는 근거: {fake or '없음'}")


===== 8번째 줄 (정답 Reconnaissance) =====
src=175.45.176.1:20549 dst=149.171.126.10:111 proto=udp service=- state=INT dur=0.000 spkts=2 dpkts=0 sbytes=168 dbytes=0 rate=2000.0 ct_dst_ltm=1 ct_src_dport_ltm=1

{
  "attack_type": "Reconnaissance",
  "mitre_tactic": "TA0043 Reconnaissance",
  "kill_chain_stage": "초반(정찰)",
  "evidence": [
    "src=175.45.176.1:20549",
    "dst=149.171.126.10:111",
    "proto=udp",
    "service=-",
    "state=INT",
    "dur=0.000",
    "spkts=2",
    "dpkts=0",
    "sbytes=168",
    "dbytes=0",
    "ct_dst_ltm=1",
    "ct_src_dport_ltm=1"
  ],
  "reason": "UDP 111 포트로 소량 패킷을 보냈으나 응답이 없어 서비스 식별 없이 포트 확인성 탐지 시도로 판단됩니다.",
  "recommended_action": "출발지 IP와 동일 목적지 포트에 대한 추가 시도 여부를 모니터링하고, 불필요한 UDP 111 접근은 방화벽에서 차단하십시오."
}

→ 예측 Reconnaissance / 정답 Reconnaissance | TA0043 Reconnaissance | 초반(정찰)
→ 로그에 없는 근거: 없음

===== 12번째 줄 (정답 Exploits) =====
src=175.45.176.1:63277 dst=149.171.126.18:110 proto=tcp service=pop3 state=FIN dur=1.272 spkts=34 dpkts=116 sbytes=1546 dbyt

### 토론
- 근거(evidence)가 로그에 그대로 있는지, 지어낸 값은 없는지 봅니다. 로그에 없는 근거가 있으면 그 설명은 믿기 어렵습니다.
- 유형은 맞혔지만 근거가 엉뚱한 경우, 유형은 틀렸지만 근거는 맞는 경우를 구분합니다.
- 심화: 06s(SHAP)가 만든 근거 문장을 프롬프트에 넣으면 지어낸 근거가 줄어드는지 비교합니다.